In [ ]:
!nvidia-smi

import torch
print(f"\n PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

/bin/bash: line 1: nvidia-smi: command not found

🔥 PyTorch version: 2.8.0+cu126
✅ CUDA available: False


In [ ]:
%%time
%%capture

print(" 开始安装依赖...")

# 安装核心库 - 使用最新稳定的兼容版本
!pip install torch==2.0.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 关键修复：使用支持 Llama 3.1 的 transformers 版本
# transformers 4.43.0+ 完全支持 Llama 3.1 的新 rope_scaling 格式
!pip install transformers==4.43.0
!pip install accelerate==0.31.0
!pip install peft==0.11.0
!pip install datasets==2.14.0

# 其他依赖
!pip install numpy==1.24.3
!pip install wandb==0.15.8
!pip install rouge-score
!pip install scikit-learn
!pip install sentencepiece
!pip install protobuf

print(" 所有依赖安装完成！")
print(" 版本信息：")
print("  - PyTorch: 2.0.1+cu118")
print("  - Transformers: 4.43.0 (支持 Llama 3.1)")
print("  - Accelerate: 0.31.0")
print("  - PEFT: 0.11.0")
print("  - Datasets: 2.14.0")
print("\n 这些版本完全支持 Llama 3.1 和 DeepSeek 模型！")

CPU times: user 16 s, sys: 2.35 s, total: 18.3 s
Wall time: 1min 3s


In [ ]:
from google.colab import files
import zipfile
import os

print(" 请上传训练文件包...")
print("=" * 50)
print(" 需要上传: fingpt_complete.zip")
print()
print("  这个包包含:")
print("  - 训练脚本 (train_lora.py, utils.py, prompt.py)")
print("  - 配置文件 (config.json, .env)")
print("  - DOW30 数据集 (1,230 训练 + 300 测试)")
print()
print(" 如果还没有创建这个文件，请在本地执行:")
print("cd '/Users/tiantian/Downloads/CU Material/S2/AI4Finance/Assignment /Assignment2/Test/FinGPT_Forecaster'")
print("zip -r fingpt_complete.zip train_lora.py utils.py prompt.py config.json .env data/fingpt-forecaster-dow30-202305-202405/")
print("=" * 50)
print("\n 等待上传...")

uploaded = files.upload()

print("\n 文件上传完成！")

# 解压
zip_file = list(uploaded.keys())[0]
print(f" 解压 {zip_file}...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('.')

print("\n 文件解压完成！")
print("\n 当前目录结构:")
!ls -lh
print("\n 数据集目录:")
!ls -lh data/fingpt-forecaster-dow30-202305-202405/

📦 请上传训练文件包...
📁 需要上传: fingpt_complete.zip

⚠️  这个包包含:
  - 训练脚本 (train_lora.py, utils.py, prompt.py)
  - 配置文件 (config.json, .env)
  - DOW30 数据集 (1,230 训练 + 300 测试)

💡 如果还没有创建这个文件，请在本地执行:
cd '/Users/tiantian/Downloads/CU Material/S2/AI4Finance/Assignment /Assignment2/Test/FinGPT_Forecaster'
zip -r fingpt_complete.zip train_lora.py utils.py prompt.py config.json .env data/fingpt-forecaster-dow30-202305-202405/

⏳ 等待上传...


Saving fingpt_complete.zip to fingpt_complete.zip

✅ 文件上传完成！
📂 解压 fingpt_complete.zip...

✅ 文件解压完成！

📁 当前目录结构:
total 2.5M
-rw-r--r-- 1 root root  784 Nov  4 19:19 config.json
drwxr-xr-x 3 root root 4.0K Nov  4 19:19 data
-rw-r--r-- 1 root root 2.4M Nov  4 19:19 fingpt_complete.zip
-rw-r--r-- 1 root root 7.0K Nov  4 19:19 prompt.py
drwxr-xr-x 1 root root 4.0K Nov  3 14:39 sample_data
-rw-r--r-- 1 root root 9.0K Nov  4 19:19 train_lora.py
-rw-r--r-- 1 root root 6.4K Nov  4 19:19 utils.py

📁 数据集目录:
total 12K
-rw-r--r-- 1 root root   29 Nov  4 19:19 dataset_dict.json
drwxr-xr-x 2 root root 4.0K Nov  4 19:19 test
drwxr-xr-x 2 root root 4.0K Nov  4 19:19 train


In [ ]:
from huggingface_hub import login
import wandb
import os

# 读取 .env 文件
if os.path.exists('.env'):
    print(" 读取 .env 文件...")
    with open('.env', 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                if '=' in line:
                    key, value = line.strip().split('=', 1)
                    os.environ[key] = value
                    print(f"  ✓ {key}")
else:
    print(" 未找到 .env 文件，请手动输入 API Keys:")
    os.environ['HF_TOKEN'] = input("HuggingFace Token: ")
    os.environ['WANDB_API_KEY'] = input("Wandb API Key: ")

# 登录 HuggingFace
print("\n 登录 HuggingFace...")
login(token=os.environ['HF_TOKEN'])

# 登录 Wandb
print(" 登录 Wandb...")
wandb.login(key=os.environ['WANDB_API_KEY'])

print("\n API 登录成功！")

📄 读取 .env 文件...
  ✓ FINNHUB_KEY
  ✓ OPENAI_KEY
  ✓ HF_TOKEN
  ✓ WANDB_API_KEY
  ✓ WANDB_PROJECT

🤗 登录 HuggingFace...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


📊 登录 Wandb...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: th3166 (th3166-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✅ API 登录成功！


In [ ]:
%%time
from datasets import load_from_disk, DatasetDict
import os

print(" 加载本地 DOW30 数据集...")
print("=" * 50)

# 检查数据集目录是否存在
dataset_path = "./data/fingpt-forecaster-dow30-202305-202405"

if not os.path.exists(dataset_path):
    print(" 错误：未找到本地数据集！")
    print(f"预期路径: {dataset_path}")
    print("\n请确保你已经上传了包含数据集的 zip 文件。")
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

print(f"✓ 找到数据集目录: {dataset_path}")

# 加载数据集
print("\n 读取数据集...")
try:
    # 方法1: 尝试使用 load_from_disk
    dataset = load_from_disk(dataset_path)
    print(" 使用 load_from_disk 加载成功！")
except Exception as e:
    print(f" load_from_disk 失败: {e}")
    print(" 尝试手动加载...")

    # 方法2: 手动从 arrow 文件加载
    from datasets import Dataset
    import pyarrow as pa

    train_arrow = os.path.join(dataset_path, "train", "data-00000-of-00001.arrow")
    test_arrow = os.path.join(dataset_path, "test", "data-00000-of-00001.arrow")

    if os.path.exists(train_arrow) and os.path.exists(test_arrow):
        # 读取 arrow 文件
        with pa.memory_map(train_arrow, 'r') as source:
            train_table = pa.ipc.open_stream(source).read_all()
        with pa.memory_map(test_arrow, 'r') as source:
            test_table = pa.ipc.open_stream(source).read_all()

        # 转换为 Dataset
        train_dataset = Dataset(train_table)
        test_dataset = Dataset(test_table)

        # 创建 DatasetDict
        dataset = DatasetDict({
            'train': train_dataset,
            'test': test_dataset
        })
        print(" 从 Arrow 文件加载成功！")
    else:
        raise FileNotFoundError("Arrow files not found!")

print(f"\n 数据集加载完成！")
print(f"训练集: {len(dataset['train'])} 样本")
print(f"测试集: {len(dataset['test'])} 样本")
print(f"\n示例数据:")
print(dataset['train'][0])

📥 加载本地 DOW30 数据集...
✓ 找到数据集目录: ./data/fingpt-forecaster-dow30-202305-202405

📖 读取数据集...
⚠️ load_from_disk 失败: Protocol not known: ./data/fingpt-forecaster-dow30-202305-202405
🔄 尝试手动加载...
✅ 从 Arrow 文件加载成功！

✅ 数据集加载完成！
训练集: 1230 样本
测试集: 300 样本

示例数据:
{'prompt': "[INST]<<SYS>>\nYou are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]\nPrediction: ...\nAnalysis: ...\n\n<</SYS>>\n\n[Company Introduction]:\n\nAmerican Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of today

In [ ]:
%%time
from google.colab import drive
import glob
import shutil
import os

print("=" * 60)
print(" 开始训练 Llama-3.1-8B")
print("=" * 60)

print("\n  训练配置:")
print("   Max Length: 2048")
print("   LoRA: r=4, alpha=8")
print("   Epochs: 3")
print("   Batch Size: 1")
print("   Gradient Accumulation: 16")
print("   预计时间: 40-60 分钟")
print("=" * 60)

print(f"\n监控地址: https://wandb.ai/th3166/fingpt-forecaster-assignment2")
print("=" * 60)
print()

# ============================================================
# 训练命令
# ============================================================
# 第一次训练
#!python train_lora.py \
#     --run_name assignment2-llama31-dow30 \
#     --base_model llama31 \
#     --dataset /content/data/fingpt-forecaster-dow30-202305-202405 \
#     --max_length 2048 \
#     --batch_size 1 \
#     --gradient_accumulation_steps 16 \
#     --learning_rate 5e-5 \
#     --num_epochs 3 \
#     --log_interval 10 \
#     --warmup_ratio 0.03 \
#     --scheduler constant \
#     --evaluation_strategy steps \
#     --eval_steps 0.1
#
#第二次训练
!python train_lora.py \
    --run_name assignment2-llama31-dow30 \
    --base_model llama31 \
    --dataset /content/data/fingpt-forecaster-dow30-202305-202405 \
    --max_length 2048 \
    --batch_size 1 \
    --gradient_accumulation_steps 16 \
    --learning_rate 2e-5 \
    --num_epochs 5 \
    --lora_r 8 \
    --lora_alpha 16 \
    --log_interval 10 \
    --warmup_ratio 0.1 \
    --scheduler cosine \
    --evaluation_strategy steps \
    --eval_steps 0.1

print("\n" + "=" * 60)
print("✅ Llama-3.1-8B 训练完成！")
print("=" * 60)

# ============================================================
# 自动保存到 Google Drive
# ============================================================
print("\n" + "=" * 60)
print(" 自动保存模型到 Google Drive...")
print("=" * 60)

# 挂载 Google Drive（如果还没挂载）
if not os.path.exists('/content/drive'):
    print(" 挂载 Google Drive...")
    drive.mount('/content/drive')
else:
    print("✓ Google Drive 已挂载")

# 创建保存目录
drive_path = '/content/drive/MyDrive/Assignment2_Models'
os.makedirs(drive_path, exist_ok=True)
print(f"✓ 保存目录: {drive_path}")

# 找到刚训练的模型
llama_models = sorted(glob.glob('finetuned_models/assignment2-llama31-dow30_*'))
if llama_models:
    llama_model = llama_models[-1]
    print(f"\n✓ 找到模型: {llama_model}")

    # 复制到 Google Drive
    model_name = os.path.basename(llama_model)
    target_path = f"{drive_path}/{model_name}"

    if os.path.exists(target_path):
        print(f"  模型已存在，跳过: {model_name}")
    else:
        print(f" 正在复制到 Google Drive...")
        shutil.copytree(llama_model, target_path)
        print(f" Llama-3.1 模型已保存")
        print(f"   路径: {target_path}")
else:
    print(" 未找到训练好的模型！")

print("\n" + "=" * 60)
print(" Llama-3.1-8B 训练和保存完成！")
print("=" * 60)

🚀 开始训练 Llama-3.1-8B

⚙️  训练配置:
   Max Length: 2048
   LoRA: r=4, alpha=8
   Epochs: 3
   Batch Size: 1
   Gradient Accumulation: 16
   预计时间: 40-60 分钟

监控地址: https://wandb.ai/th3166/fingpt-forecaster-assignment2

2025-11-04 19:20:30.543330: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 19:20:30.559961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762284030.579043    6211 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762284030.585356    6211 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to reg

ValueError: mount failed

In [ ]:
# ============================================================
# 自动保存到 Google Drive
# ============================================================
print("\n" + "=" * 60)
print(" 自动保存模型到 Google Drive...")
print("=" * 60)

# 挂载 Google Drive（如果还没挂载）
if not os.path.exists('/content/drive'):
    print(" 挂载 Google Drive...")
    drive.mount('/content/drive')
else:
    print("✓ Google Drive 已挂载")

# 创建保存目录
drive_path = '/content/drive/MyDrive/Assignment2_Models'
os.makedirs(drive_path, exist_ok=True)
print(f"✓ 保存目录: {drive_path}")

# 找到刚训练的模型
llama_models = sorted(glob.glob('finetuned_models/assignment2-llama31-dow30_*'))
if llama_models:
    llama_model = llama_models[-1]
    print(f"\n✓ 找到模型: {llama_model}")

    # 复制到 Google Drive
    model_name = os.path.basename(llama_model)
    target_path = f"{drive_path}/{model_name}"

    if os.path.exists(target_path):
        print(f"  模型已存在，跳过: {model_name}")
    else:
        print(f" 正在复制到 Google Drive...")
        shutil.copytree(llama_model, target_path)
        print(f" Llama-3.1 模型已保存")
        print(f"   路径: {target_path}")
else:
    print(" 未找到训练好的模型！")

print("\n" + "=" * 60)
print(" Llama-3.1-8B 训练和保存完成！")
print("=" * 60)


💾 自动保存模型到 Google Drive...
📁 挂载 Google Drive...
Mounted at /content/drive
✓ 保存目录: /content/drive/MyDrive/Assignment2_Models

✓ 找到模型: finetuned_models/assignment2-llama31-dow30_202511041921
📤 正在复制到 Google Drive...
✅ Llama-3.1 模型已保存
   路径: /content/drive/MyDrive/Assignment2_Models/assignment2-llama31-dow30_202511041921

🎉 Llama-3.1-8B 训练和保存完成！


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%time
import glob
import shutil
import os

print("=" * 60)
print(" 开始训练 DeepSeek-R1-Distill-Llama-8B")
print("=" * 60)

print("\n  训练配置:")
print("   Max Length: 2048")
print("   LoRA: r=4, alpha=8")
print("   Epochs: 3")
print("   Batch Size: 1")
print("   Gradient Accumulation: 16")
print("   预计时间: 40-60 分钟")
print("=" * 60)

print(f"\n监控地址: https://wandb.ai/th3166/fingpt-forecaster-assignment2")
print("=" * 60)
print()

# ============================================================
# 训练命令
# ============================================================
# 第一次训练
#!python train_lora.py \
#     --run_name assignment2-deepseek-dow30 \
#     --base_model deepseek \
#     --dataset /content/data/fingpt-forecaster-dow30-202305-202405 \
#     --max_length 2048 \
#     --batch_size 1 \
#     --gradient_accumulation_steps 16 \
#     --learning_rate 5e-5 \
#     --num_epochs 3 \
#     --log_interval 10 \
#     --warmup_ratio 0.03 \
#     --scheduler constant \
#     --evaluation_strategy steps \
#     --eval_steps 0.1

#第二次训练
!python train_lora.py \
    --run_name assignment2-deepseek-dow30 \
    --base_model deepseek \
    --dataset /content/data/fingpt-forecaster-dow30-202305-202405 \
    --max_length 2048 \
    --batch_size 1 \
    --gradient_accumulation_steps 16 \
    --learning_rate 3e-5 \
    --num_epochs 4 \
    --lora_r 6 \
    --lora_alpha 12 \
    --log_interval 10 \
    --warmup_ratio 0.05 \
    --scheduler cosine \
    --evaluation_strategy steps \
    --eval_steps 0.1

print("\n" + "=" * 60)
print(" DeepSeek-R1 训练完成！")
print("=" * 60)

# ============================================================
# 自动保存到 Google Drive
# ============================================================
print("\n" + "=" * 60)
print(" 自动保存模型到 Google Drive...")
print("=" * 60)

# Google Drive 应该已经在 Step 6 挂载了
drive_path = '/content/drive/MyDrive/Assignment2_Models'
print(f"✓ 保存目录: {drive_path}")

# 找到刚训练的模型
deepseek_models = sorted(glob.glob('finetuned_models/assignment2-deepseek-dow30_*'))
if deepseek_models:
    deepseek_model = deepseek_models[-1]
    print(f"\n✓ 找到模型: {deepseek_model}")

    # 复制到 Google Drive
    model_name = os.path.basename(deepseek_model)
    target_path = f"{drive_path}/{model_name}"

    if os.path.exists(target_path):
        print(f"  模型已存在，跳过: {model_name}")
    else:
        print(f"📤 正在复制到 Google Drive...")
        shutil.copytree(deepseek_model, target_path)
        print(f" DeepSeek 模型已保存")
        print(f"   路径: {target_path}")
else:
    print(" 未找到训练好的模型！")

print("\n" + "=" * 60)
print(" DeepSeek-R1 训练和保存完成！")
print("=" * 60)

🚀 开始训练 DeepSeek-R1-Distill-Llama-8B

⚙️  训练配置:
   Max Length: 2048
   LoRA: r=4, alpha=8
   Epochs: 3
   Batch Size: 1
   Gradient Accumulation: 16
   预计时间: 40-60 分钟

监控地址: https://wandb.ai/th3166/fingpt-forecaster-assignment2

2025-11-04 22:15:24.180271: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 22:15:24.198220: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762294524.219650   51917 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762294524.226175   51917 cuda_blas.cc:1407] Unable to register cuBLAS factory: A

In [ ]:
%%time
import os
import torch
import re
import time
import json
import pickle
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import PeftModel
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, mean_squared_error
from tqdm import tqdm
from utils import *

print("📦 导入评估所需的库...")
print("=" * 60)

# 创建结果保存目录
os.makedirs("./comparison_results", exist_ok=True)
print("✅ 结果保存目录已创建: ./comparison_results/")

# 检查 GPU 可用性
print(f"\n🎮 GPU 状态:")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✅ 评估环境准备完成！")
print("=" * 60)

📦 导入评估所需的库...
✅ 结果保存目录已创建: ./comparison_results/

🎮 GPU 状态:
   CUDA available: True
   GPU: NVIDIA A100-SXM4-80GB
   GPU Memory: 85.17 GB

✅ 评估环境准备完成！
CPU times: user 5.29 s, sys: 617 ms, total: 5.91 s
Wall time: 4.63 s


In [ ]:
%%time
import glob

print("🔍 查找微调后的模型...")
print("=" * 60)

# 找到微调后的模型路径
llama_models = sorted(glob.glob('finetuned_models/assignment2-llama31-dow30_*'))
deepseek_models = sorted(glob.glob('finetuned_models/assignment2-deepseek-dow30_*'))

if not llama_models:
    print("❌ 未找到 Llama-3.1 微调模型！请先完成训练。")
    raise FileNotFoundError("Llama-3.1 fine-tuned model not found")

if not deepseek_models:
    print("❌ 未找到 DeepSeek 微调模型！请先完成训练。")
    raise FileNotFoundError("DeepSeek fine-tuned model not found")

llama_finetuned_path = llama_models[-1]
deepseek_finetuned_path = deepseek_models[-1]

print(f"✅ 找到 Llama-3.1 模型: {llama_finetuned_path}")
print(f"✅ 找到 DeepSeek 模型: {deepseek_finetuned_path}")
print("=" * 60)

# ============================================================
# 加载 Llama-3.1 Base 模型
# ============================================================
print("\n📥 加载 Llama-3.1-8B Base 模型...")
print("   (这可能需要几分钟...)")

# 修复 Llama 3.1 的 rope_scaling 配置
llama_config = AutoConfig.from_pretrained('meta-llama/Llama-3.1-8B', trust_remote_code=True)

# 关键修复：处理 Llama 3.1 的 rope_scaling
if hasattr(llama_config, 'rope_scaling') and llama_config.rope_scaling is not None:
    if 'rope_type' in llama_config.rope_scaling and 'type' not in llama_config.rope_scaling:
        # Llama 3.1 使用 rope_type，但 transformers 期望 type
        llama_config.rope_scaling['type'] = llama_config.rope_scaling.get('rope_type', 'default')
        print("   ✓ 已修复 Llama 3.1 rope_scaling 配置")

llama3_base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-3.1-8B',
    config=llama_config,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16,
)
llama3_base_model.eval()
print("✅ Llama-3.1 Base 模型加载完成")

# ============================================================
# 加载 Llama-3.1 Fine-tuned 模型
# ============================================================
print("\n📥 加载 Llama-3.1 Fine-tuned 模型...")

llama3_model = PeftModel.from_pretrained(
    llama3_base_model,
    llama_finetuned_path,
    torch_dtype=torch.float16,
)
llama3_model.eval()
print("✅ Llama-3.1 Fine-tuned 模型加载完成")

# ============================================================
# 加载 DeepSeek Base 模型
# ============================================================
print("\n📥 加载 DeepSeek-R1-Distill-Llama-8B Base 模型...")
print("   (这可能需要几分钟...)")

# 修复 DeepSeek 的 rope_scaling 配置（与 Llama 3.1 类似）
deepseek_config = AutoConfig.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Llama-8B', trust_remote_code=True)

# 关键修复：处理 rope_scaling
if hasattr(deepseek_config, 'rope_scaling') and deepseek_config.rope_scaling is not None:
    if 'rope_type' in deepseek_config.rope_scaling and 'type' not in deepseek_config.rope_scaling:
        # DeepSeek 也可能使用 rope_type，但 transformers 期望 type
        deepseek_config.rope_scaling['type'] = deepseek_config.rope_scaling.get('rope_type', 'default')
        print("   ✓ 已修复 DeepSeek rope_scaling 配置")

deepseek_base_model = AutoModelForCausalLM.from_pretrained(
    'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    config=deepseek_config,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16,
)
deepseek_base_model.eval()
print("✅ DeepSeek Base 模型加载完成")

# ============================================================
# 加载 DeepSeek Fine-tuned 模型
# ============================================================
print("\n📥 加载 DeepSeek Fine-tuned 模型...")

deepseek_model = PeftModel.from_pretrained(
    deepseek_base_model,
    deepseek_finetuned_path,
    torch_dtype=torch.float16,
)
deepseek_model.eval()
print("✅ DeepSeek Fine-tuned 模型加载完成")

# ============================================================
# 加载 Tokenizers
# ============================================================
print("\n📥 加载 Tokenizers...")

llama3_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B')
llama3_tokenizer.padding_side = "right"
llama3_tokenizer.pad_token_id = llama3_tokenizer.eos_token_id

deepseek_tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/DeepSeek-R1-Distill-Llama-8B')
deepseek_tokenizer.padding_side = "right"
deepseek_tokenizer.pad_token_id = deepseek_tokenizer.eos_token_id

print("✅ Tokenizers 加载完成")

print("\n" + "=" * 60)
print("🎉 所有模型加载完成！")
print("=" * 60)
print(f"\n📊 已加载的模型:")
print(f"   1. Llama-3.1 Base")
print(f"   2. Llama-3.1 Fine-tuned ({os.path.basename(llama_finetuned_path)})")
print(f"   3. DeepSeek Base")
print(f"   4. DeepSeek Fine-tuned ({os.path.basename(deepseek_finetuned_path)})")
print("=" * 60)

🔍 查找微调后的模型...
✅ 找到 Llama-3.1 模型: finetuned_models/assignment2-llama31-dow30_202511041921
✅ 找到 DeepSeek 模型: finetuned_models/assignment2-deepseek-dow30_202511042216

📥 加载 Llama-3.1-8B Base 模型...
   (这可能需要几分钟...)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   ✓ 已修复 Llama 3.1 rope_scaling 配置


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Llama-3.1 Base 模型加载完成

📥 加载 Llama-3.1 Fine-tuned 模型...
✅ Llama-3.1 Fine-tuned 模型加载完成

📥 加载 DeepSeek-R1-Distill-Llama-8B Base 模型...
   (这可能需要几分钟...)
   ✓ 已修复 DeepSeek rope_scaling 配置


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ DeepSeek Base 模型加载完成

📥 加载 DeepSeek Fine-tuned 模型...
✅ DeepSeek Fine-tuned 模型加载完成

📥 加载 Tokenizers...
✅ Tokenizers 加载完成

🎉 所有模型加载完成！

📊 已加载的模型:
   1. Llama-3.1 Base
   2. Llama-3.1 Fine-tuned (assignment2-llama31-dow30_202511041921)
   3. DeepSeek Base
   4. DeepSeek Fine-tuned (assignment2-deepseek-dow30_202511042216)
CPU times: user 44.1 s, sys: 18.2 s, total: 1min 2s
Wall time: 21.6 s


In [ ]:
print("📥 加载测试数据集...")
print("=" * 60)

# 加载数据集 - 尝试多种路径格式
import os

# 尝试多个可能的路径
possible_paths = [
    "data/fingpt-forecaster-dow30-202305-202405",
    "./data/fingpt-forecaster-dow30-202305-202405",
    "/content/data/fingpt-forecaster-dow30-202305-202405"
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        print(f"✓ 找到数据集: {dataset_path}")
        break

if dataset_path is None:
    raise FileNotFoundError(f"数据集未找到！请检查数据是否已上传。尝试过的路径: {possible_paths}")

# 加载数据集 - 使用 file:// 协议或相对路径
try:
    # 优先使用相对路径格式（不带 ./ 或 /）
    if dataset_path.startswith("/content/"):
        # 转换为相对路径
        relative_path = dataset_path.replace("/content/", "")
        print(f"   使用相对路径: {relative_path}")
        full_dataset = load_from_disk(relative_path)
    else:
        full_dataset = load_from_disk(dataset_path)

    test_dataset = full_dataset['test']

    print(f"\n✅ 测试集加载完成！")
    print(f"   样本数量: {len(test_dataset)}")
    print(f"   字段: {test_dataset.column_names}")

    # 显示一个示例
    print(f"\n📄 测试样本示例:")
    print(f"   Prompt 长度: {len(test_dataset[0]['prompt'])} 字符")
    print(f"   Answer: {test_dataset[0]['answer'][:100]}...")

except Exception as e:
    print(f"❌ 加载失败: {e}")
    print(f"💡 尝试手动从 Arrow 文件加载...")

    # 备用方案：手动从 arrow 文件加载
    from datasets import Dataset, DatasetDict
    import pyarrow as pa

    test_arrow = os.path.join(dataset_path, "test", "data-00000-of-00001.arrow")

    if os.path.exists(test_arrow):
        with pa.memory_map(test_arrow, 'r') as source:
            test_table = pa.ipc.open_stream(source).read_all()

        test_dataset = Dataset(test_table)
        print(f"\n✅ 测试集加载完成（从 Arrow 文件）！")
        print(f"   样本数量: {len(test_dataset)}")
        print(f"   字段: {test_dataset.column_names}")
    else:
        raise FileNotFoundError(f"Arrow 文件未找到: {test_arrow}")

print("=" * 60)

📥 加载测试数据集...
✓ 找到数据集: data/fingpt-forecaster-dow30-202305-202405
❌ 加载失败: Protocol not known: data/fingpt-forecaster-dow30-202305-202405
💡 尝试手动从 Arrow 文件加载...

✅ 测试集加载完成（从 Arrow 文件）！
   样本数量: 300
   字段: ['prompt', 'answer', 'period', 'label', 'symbol']


In [ ]:
print("🔧 定义评估函数...")
print("=" * 60)

# ============================================================
# 函数 1: 单个推理测试（带时间测量）
# ============================================================
def test_demo(model, tokenizer, prompt):
    """
    对单个 prompt 进行推理，返回答案和推理时间
    """
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        padding=False,
        max_length=4096,
        truncation=True
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    start_time = time.time()
    res = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    end_time = time.time()

    output = tokenizer.decode(res[0], skip_special_tokens=True)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL)

    return answer, end_time - start_time

# ============================================================
# 函数 2: 批量测试函数
# ============================================================
def test_acc(test_dataset, modelname, max_samples=None):
    """
    批量评估模型性能

    Args:
        test_dataset: 测试数据集
        modelname: 'llama3' 或 'deepseek'
        max_samples: 最大测试样本数（None = 全部）

    Returns:
        answers_base, answers_fine_tuned, gts, times_base, times_fine_tuned
    """
    answers_base, answers_fine_tuned, gts, times_base, times_fine_tuned = [], [], [], [], []

    if modelname == "llama3":
        base_model = llama3_base_model
        model = llama3_model
        tokenizer = llama3_tokenizer
    elif modelname == "deepseek":
        base_model = deepseek_base_model
        model = deepseek_model
        tokenizer = deepseek_tokenizer
    else:
        raise ValueError(f"Unknown model name: {modelname}")

    # 限制样本数量（如果指定）
    num_samples = len(test_dataset) if max_samples is None else min(max_samples, len(test_dataset))

    print(f"\n🚀 开始评估 {modelname.upper()} 模型...")
    print(f"   样本数量: {num_samples}")
    print(f"   Base 模型: {modelname} (未微调)")
    print(f"   Fine-tuned 模型: {modelname} (LoRA 微调)")
    print("=" * 60)

    for i in tqdm(range(num_samples), desc=f"评估 {modelname}"):
        try:
            prompt = test_dataset[i]['prompt']
            gt = test_dataset[i]['answer']

            # 测试 Base 模型
            answer_base, time_base = test_demo(base_model, tokenizer, prompt)

            # 测试 Fine-tuned 模型
            answer_fine_tuned, time_fine_tuned = test_demo(model, tokenizer, prompt)

            answers_base.append(answer_base)
            answers_fine_tuned.append(answer_fine_tuned)
            gts.append(gt)
            times_base.append(time_base)
            times_fine_tuned.append(time_fine_tuned)

        except Exception as e:
            print(f"\n⚠️  样本 {i} 处理失败: {e}")
            continue

    return answers_base, answers_fine_tuned, gts, times_base, times_fine_tuned

print("✅ 评估函数定义完成！")
print("=" * 60)

🔧 定义评估函数...
✅ 评估函数定义完成！


In [ ]:
%%time

print("=" * 60)
print("🧪 开始评估 Llama-3.1-8B")
print("=" * 60)

# ⚠️ 优化配置：评估 35 个样本（快速测试）
# 如果想评估更多样本，可以修改这个值
MAX_SAMPLES = 300  # 优化配置：35 个样本

llama3_answers_base, llama3_answers_fine_tuned, llama3_gts, llama3_base_times, llama3_fine_tuned_times = test_acc(
    test_dataset,
    "llama3",
    max_samples=MAX_SAMPLES
)

print("\n" + "=" * 60)
print("✅ Llama-3.1 评估完成！")
print("=" * 60)
print(f"📊 评估结果:")
print(f"   Base 模型推理次数: {len(llama3_base_times)}")
print(f"   Fine-tuned 模型推理次数: {len(llama3_fine_tuned_times)}")
print(f"   平均推理时间 (Base): {sum(llama3_base_times)/len(llama3_base_times):.2f}s")
print(f"   平均推理时间 (Fine-tuned): {sum(llama3_fine_tuned_times)/len(llama3_fine_tuned_times):.2f}s")
print("=" * 60)

🧪 开始评估 Llama-3.1-8B

🚀 开始评估 LLAMA3 模型...
   样本数量: 300
   Base 模型: llama3 (未微调)
   Fine-tuned 模型: llama3 (LoRA 微调)


评估 llama3: 100%|██████████| 300/300 [4:05:33<00:00, 49.11s/it]


✅ Llama-3.1 评估完成！
📊 评估结果:
   Base 模型推理次数: 300
   Fine-tuned 模型推理次数: 300
   平均推理时间 (Base): 24.08s
   平均推理时间 (Fine-tuned): 25.01s
CPU times: user 4h 5min 29s, sys: 8.85 s, total: 4h 5min 38s
Wall time: 4h 5min 33s


## 📈 Step 14: 计算 Llama-3.1 性能指标

In [ ]:
print("📊 计算 Llama-3.1 性能指标...")
print("=" * 60)

# 计算 Base 模型指标
llama3_base_metrics = calc_metrics(llama3_answers_base, llama3_gts)
print("\n🔵 Llama-3.1 Base 模型:")
print(f"   Valid Count: {llama3_base_metrics.get('valid_count', 0)}")
print(f"   Binary Accuracy: {llama3_base_metrics.get('bin_acc', 0):.4f}")
print(f"   MSE: {llama3_base_metrics.get('mse', 0):.4f}")
print(f"   ROUGE-1 (Positive): {llama3_base_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Concerns): {llama3_base_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Analysis): {llama3_base_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}")

# 计算 Fine-tuned 模型指标
llama3_fine_tuned_metrics = calc_metrics(llama3_answers_fine_tuned, llama3_gts)
print("\n🟢 Llama-3.1 Fine-tuned 模型:")
print(f"   Valid Count: {llama3_fine_tuned_metrics.get('valid_count', 0)}")
print(f"   Binary Accuracy: {llama3_fine_tuned_metrics.get('bin_acc', 0):.4f}")
print(f"   MSE: {llama3_fine_tuned_metrics.get('mse', 0):.4f}")
print(f"   ROUGE-1 (Positive): {llama3_fine_tuned_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Concerns): {llama3_fine_tuned_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Analysis): {llama3_fine_tuned_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}")

# 保存结果
with open("./comparison_results/llama3_base_metrics.pkl", "wb") as f:
    pickle.dump(llama3_base_metrics, f)

with open("./comparison_results/llama3_fine_tuned_metrics.pkl", "wb") as f:
    pickle.dump(llama3_fine_tuned_metrics, f)

with open("./comparison_results/llama3_base_times.pkl", "wb") as f:
    pickle.dump(llama3_base_times, f)

with open("./comparison_results/llama3_fine_tuned_times.pkl", "wb") as f:
    pickle.dump(llama3_fine_tuned_times, f)

print("\n✅ Llama-3.1 指标已保存到 ./comparison_results/")
print("=" * 60)

📊 计算 Llama-3.1 性能指标...

Binary Accuracy: 0.51  |  Mean Square Error: 11.93

Rouge Score of Positive Developments: {'rouge1': 0.41438824927674933, 'rouge2': 0.14240269029178884, 'rougeL': 0.24837721262779947}

Rouge Score of Potential Concerns: {'rouge1': 0.39136633294476614, 'rouge2': 0.12296092840395334, 'rougeL': 0.23960082208671754}

Rouge Score of Summary Analysis: {'rouge1': 0.4171927526799803, 'rouge2': 0.12158616628515073, 'rougeL': 0.21529519766949215}

🔵 Llama-3.1 Base 模型:
   Valid Count: 283
   Binary Accuracy: 0.5124
   MSE: 11.9293
   ROUGE-1 (Positive): 0.4144
   ROUGE-1 (Concerns): 0.3914
   ROUGE-1 (Analysis): 0.4172

Binary Accuracy: 0.49  |  Mean Square Error: 11.94

Rouge Score of Positive Developments: {'rouge1': 0.4123559347206689, 'rouge2': 0.1421987452822163, 'rougeL': 0.24876837725711848}

Rouge Score of Potential Concerns: {'rouge1': 0.3928308159690958, 'rouge2': 0.12298030111390075, 'rougeL': 0.23835592675833805}

Rouge Score of Summary Analysis: {'rouge1': 0.4

In [ ]:
%%time

print("=" * 60)
print("🧪 开始评估 DeepSeek-R1-Distill-Llama-8B")
print("=" * 60)

# ⚠️ 优化配置：评估 35 个样本（快速测试）
MAX_SAMPLES = 150  # 优化配置：35 个样本

deepseek_answers_base, deepseek_answers_fine_tuned, deepseek_gts, deepseek_base_times, deepseek_fine_tuned_times = test_acc(
    test_dataset,
    "deepseek",
    max_samples=MAX_SAMPLES
)

print("\n" + "=" * 60)
print("✅ DeepSeek 评估完成！")
print("=" * 60)
print(f"📊 评估结果:")
print(f"   Base 模型推理次数: {len(deepseek_base_times)}")
print(f"   Fine-tuned 模型推理次数: {len(deepseek_fine_tuned_times)}")
print(f"   平均推理时间 (Base): {sum(deepseek_base_times)/len(deepseek_base_times):.2f}s")
print(f"   平均推理时间 (Fine-tuned): {sum(deepseek_fine_tuned_times)/len(deepseek_fine_tuned_times):.2f}s")
print("=" * 60)

# 计算指标
print("\n📊 计算 DeepSeek 性能指标...")
print("=" * 60)

# 计算 Base 模型指标
deepseek_base_metrics = calc_metrics(deepseek_answers_base, deepseek_gts)
print("\n🔵 DeepSeek Base 模型:")
print(f"   Valid Count: {deepseek_base_metrics.get('valid_count', 0)}")
print(f"   Binary Accuracy: {deepseek_base_metrics.get('bin_acc', 0):.4f}")
print(f"   MSE: {deepseek_base_metrics.get('mse', 0):.4f}")
print(f"   ROUGE-1 (Positive): {deepseek_base_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Concerns): {deepseek_base_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Analysis): {deepseek_base_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}")

# 计算 Fine-tuned 模型指标
deepseek_fine_tuned_metrics = calc_metrics(deepseek_answers_fine_tuned, deepseek_gts)
print("\n🟢 DeepSeek Fine-tuned 模型:")
print(f"   Valid Count: {deepseek_fine_tuned_metrics.get('valid_count', 0)}")
print(f"   Binary Accuracy: {deepseek_fine_tuned_metrics.get('bin_acc', 0):.4f}")
print(f"   MSE: {deepseek_fine_tuned_metrics.get('mse', 0):.4f}")
print(f"   ROUGE-1 (Positive): {deepseek_fine_tuned_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Concerns): {deepseek_fine_tuned_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Analysis): {deepseek_fine_tuned_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}")

# 保存结果
with open("./comparison_results/deepseek_base_metrics.pkl", "wb") as f:
    pickle.dump(deepseek_base_metrics, f)

with open("./comparison_results/deepseek_fine_tuned_metrics.pkl", "wb") as f:
    pickle.dump(deepseek_fine_tuned_metrics, f)

with open("./comparison_results/deepseek_base_times.pkl", "wb") as f:
    pickle.dump(deepseek_base_times, f)

with open("./comparison_results/deepseek_fine_tuned_times.pkl", "wb") as f:
    pickle.dump(deepseek_fine_tuned_times, f)

print("\n✅ DeepSeek 指标已保存到 ./comparison_results/")
print("=" * 60)

🧪 开始评估 DeepSeek-R1-Distill-Llama-8B

🚀 开始评估 DEEPSEEK 模型...
   样本数量: 150
   Base 模型: deepseek (未微调)
   Fine-tuned 模型: deepseek (LoRA 微调)


评估 deepseek: 100%|██████████| 150/150 [2:01:10<00:00, 48.47s/it]



✅ DeepSeek 评估完成！
📊 评估结果:
   Base 模型推理次数: 150
   Fine-tuned 模型推理次数: 150
   平均推理时间 (Base): 24.34s
   平均推理时间 (Fine-tuned): 24.11s

📊 计算 DeepSeek 性能指标...

Binary Accuracy: 0.56  |  Mean Square Error: 8.54

Rouge Score of Positive Developments: {'rouge1': 0.42037485783421186, 'rouge2': 0.1472438694113917, 'rougeL': 0.24922654138594555}

Rouge Score of Potential Concerns: {'rouge1': 0.3951811059531643, 'rouge2': 0.13079588286151303, 'rougeL': 0.24031636090548103}

Rouge Score of Summary Analysis: {'rouge1': 0.43399079175258076, 'rouge2': 0.13157128784894603, 'rougeL': 0.21956513254991855}

🔵 DeepSeek Base 模型:
   Valid Count: 147
   Binary Accuracy: 0.5578
   MSE: 8.5442
   ROUGE-1 (Positive): 0.4204
   ROUGE-1 (Concerns): 0.3952
   ROUGE-1 (Analysis): 0.4340

Binary Accuracy: 0.52  |  Mean Square Error: 8.79

Rouge Score of Positive Developments: {'rouge1': 0.41863687454899995, 'rouge2': 0.1455464326012816, 'rougeL': 0.25440369948821906}

Rouge Score of Potential Concerns: {'rouge1': 0.3828

In [ ]:
import pandas as pd

print("=" * 80)
print("📊 FINAL COMPARISON REPORT")
print("=" * 80)

# ============================================================
# 1. 推理速度对比
# ============================================================
print("\n🚀 1. INFERENCE SPEED COMPARISON")
print("-" * 80)

speed_data = {
    'Model': [
        'Llama-3.1 Base',
        'Llama-3.1 Fine-tuned',
        'DeepSeek Base',
        'DeepSeek Fine-tuned'
    ],
    'Avg Time (s)': [
        f"{sum(llama3_base_times)/len(llama3_base_times):.2f}",
        f"{sum(llama3_fine_tuned_times)/len(llama3_fine_tuned_times):.2f}",
        f"{sum(deepseek_base_times)/len(deepseek_base_times):.2f}",
        f"{sum(deepseek_fine_tuned_times)/len(deepseek_fine_tuned_times):.2f}"
    ],
    'Total Time (s)': [
        f"{sum(llama3_base_times):.2f}",
        f"{sum(llama3_fine_tuned_times):.2f}",
        f"{sum(deepseek_base_times):.2f}",
        f"{sum(deepseek_fine_tuned_times):.2f}"
    ]
}

speed_df = pd.DataFrame(speed_data)
print(speed_df.to_string(index=False))

# ============================================================
# 2. 准确性对比
# ============================================================
print("\n\n📈 2. ACCURACY METRICS COMPARISON")
print("-" * 80)

accuracy_data = {
    'Model': [
        'Llama-3.1 Base',
        'Llama-3.1 Fine-tuned',
        'DeepSeek Base',
        'DeepSeek Fine-tuned'
    ],
    'Binary Acc': [
        f"{llama3_base_metrics.get('bin_acc', 0):.4f}",
        f"{llama3_fine_tuned_metrics.get('bin_acc', 0):.4f}",
        f"{deepseek_base_metrics.get('bin_acc', 0):.4f}",
        f"{deepseek_fine_tuned_metrics.get('bin_acc', 0):.4f}"
    ],
    'MSE': [
        f"{llama3_base_metrics.get('mse', 0):.2f}",
        f"{llama3_fine_tuned_metrics.get('mse', 0):.2f}",
        f"{deepseek_base_metrics.get('mse', 0):.2f}",
        f"{deepseek_fine_tuned_metrics.get('mse', 0):.2f}"
    ],
    'Valid Count': [
        f"{llama3_base_metrics.get('valid_count', 0)}",
        f"{llama3_fine_tuned_metrics.get('valid_count', 0)}",
        f"{deepseek_base_metrics.get('valid_count', 0)}",
        f"{deepseek_fine_tuned_metrics.get('valid_count', 0)}"
    ]
}

accuracy_df = pd.DataFrame(accuracy_data)
print(accuracy_df.to_string(index=False))

# ============================================================
# 3. ROUGE Scores 对比
# ============================================================
print("\n\n📝 3. ROUGE SCORES COMPARISON")
print("-" * 80)

rouge_data = {
    'Model': [
        'Llama-3.1 Base',
        'Llama-3.1 Fine-tuned',
        'DeepSeek Base',
        'DeepSeek Fine-tuned'
    ],
    'Positive R1': [
        f"{llama3_base_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{llama3_fine_tuned_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_base_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_fine_tuned_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}"
    ],
    'Concerns R1': [
        f"{llama3_base_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{llama3_fine_tuned_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_base_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_fine_tuned_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}"
    ],
    'Analysis R1': [
        f"{llama3_base_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{llama3_fine_tuned_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_base_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}",
        f"{deepseek_fine_tuned_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}"
    ]
}

rouge_df = pd.DataFrame(rouge_data)
print(rouge_df.to_string(index=False))

# ============================================================
# 4. Fine-tuned 模型直接对比
# ============================================================
print("\n\n🏆 4. FINE-TUNED MODELS HEAD-TO-HEAD COMPARISON")
print("-" * 80)

comparison_metrics = calc_metrics(llama3_answers_fine_tuned, deepseek_answers_fine_tuned)

print(f"Llama-3.1 vs DeepSeek (Fine-tuned 模型直接对比):")
print(f"   Valid Count: {comparison_metrics.get('valid_count', 0)}")
print(f"   Binary Accuracy: {comparison_metrics.get('bin_acc', 0):.4f}")
print(f"   MSE: {comparison_metrics.get('mse', 0):.4f}")
print(f"   ROUGE-1 (Positive): {comparison_metrics.get('pros_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Concerns): {comparison_metrics.get('cons_rouge_scores', {}).get('rouge1', 0):.4f}")
print(f"   ROUGE-1 (Analysis): {comparison_metrics.get('anal_rouge_scores', {}).get('rouge1', 0):.4f}")

# 保存对比指标
with open("./comparison_results/comparison_metrics.pkl", "wb") as f:
    pickle.dump(comparison_metrics, f)

print("\n✅ 对比指标已保存到 ./comparison_results/comparison_metrics.pkl")

# ============================================================
# 5. 保存完整报告
# ============================================================
print("\n📄 保存完整报告...")

full_report = {
    'speed_comparison': speed_data,
    'accuracy_comparison': accuracy_data,
    'rouge_comparison': rouge_data,
    'head_to_head': comparison_metrics,
    'llama3_base': llama3_base_metrics,
    'llama3_finetuned': llama3_fine_tuned_metrics,
    'deepseek_base': deepseek_base_metrics,
    'deepseek_finetuned': deepseek_fine_tuned_metrics
}

with open("./comparison_results/full_comparison_report.json", "w") as f:
    json.dump(full_report, f, indent=2, default=str)

print("✅ 完整报告已保存到 ./comparison_results/full_comparison_report.json")

print("\n" + "=" * 80)
print("🎉 评估完成！所有结果已保存到 ./comparison_results/")
print("=" * 80)

📊 FINAL COMPARISON REPORT

🚀 1. INFERENCE SPEED COMPARISON
--------------------------------------------------------------------------------
               Model Avg Time (s) Total Time (s)
      Llama-3.1 Base        24.08        7224.77
Llama-3.1 Fine-tuned        25.01        7502.60
       DeepSeek Base        24.34        3650.94
 DeepSeek Fine-tuned        24.11        3616.63


📈 2. ACCURACY METRICS COMPARISON
--------------------------------------------------------------------------------
               Model Binary Acc   MSE Valid Count
      Llama-3.1 Base     0.5124 11.93         283
Llama-3.1 Fine-tuned     0.4948 11.94         287
       DeepSeek Base     0.5578  8.54         147
 DeepSeek Fine-tuned     0.5203  8.79         148


📝 3. ROUGE SCORES COMPARISON
--------------------------------------------------------------------------------
               Model Positive R1 Concerns R1 Analysis R1
      Llama-3.1 Base      0.4144      0.3914      0.4172
Llama-3.1 Fine-tuned   

In [ ]:
from google.colab import files
import shutil

print("📦 打包评估结果...")
print("=" * 60)

# 创建 zip 文件
shutil.make_archive('comparison_results', 'zip', './comparison_results')

print("✅ 结果已打包为 comparison_results.zip")
print(f"   大小: {os.path.getsize('comparison_results.zip') / 1024:.2f} KB")

# 列出包含的文件
print(f"\n📁 包含的文件:")
!ls -lh comparison_results/

print("\n⏳ 开始下载...")
files.download('comparison_results.zip')

print("\n✅ 下载完成！")
print("=" * 60)
print("\n💡 文件包含:")
print("   - llama3_base_metrics.pkl")
print("   - llama3_fine_tuned_metrics.pkl")
print("   - llama3_base_times.pkl")
print("   - llama3_fine_tuned_times.pkl")
print("   - deepseek_base_metrics.pkl")
print("   - deepseek_fine_tuned_metrics.pkl")
print("   - deepseek_base_times.pkl")
print("   - deepseek_fine_tuned_times.pkl")
print("   - comparison_metrics.pkl")
print("   - full_comparison_report.json")
print("=" * 60)

📦 打包评估结果...
✅ 结果已打包为 comparison_results.zip
   大小: 7.68 KB

📁 包含的文件:
total 40K
-rw-r--r-- 1 root root  258 Nov  5 07:34 comparison_metrics.pkl
-rw-r--r-- 1 root root  258 Nov  5 07:34 deepseek_base_metrics.pkl
-rw-r--r-- 1 root root 1.4K Nov  5 07:34 deepseek_base_times.pkl
-rw-r--r-- 1 root root  258 Nov  5 07:34 deepseek_fine_tuned_metrics.pkl
-rw-r--r-- 1 root root 1.4K Nov  5 07:34 deepseek_fine_tuned_times.pkl
-rw-r--r-- 1 root root 3.9K Nov  5 07:34 full_comparison_report.json
-rw-r--r-- 1 root root  259 Nov  5 05:31 llama3_base_metrics.pkl
-rw-r--r-- 1 root root 2.7K Nov  5 05:31 llama3_base_times.pkl
-rw-r--r-- 1 root root  259 Nov  5 05:31 llama3_fine_tuned_metrics.pkl
-rw-r--r-- 1 root root 2.7K Nov  5 05:31 llama3_fine_tuned_times.pkl

⏳ 开始下载...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 下载完成！

💡 文件包含:
   - llama3_base_metrics.pkl
   - llama3_fine_tuned_metrics.pkl
   - llama3_base_times.pkl
   - llama3_fine_tuned_times.pkl
   - deepseek_base_metrics.pkl
   - deepseek_fine_tuned_metrics.pkl
   - deepseek_base_times.pkl
   - deepseek_fine_tuned_times.pkl
   - comparison_metrics.pkl
   - full_comparison_report.json


In [ ]:
from google.colab import files
import shutil
import os

print("📦 打包微调模型文件...")
print("=" * 60)

# 定义模型目录
models_dir = "/content/finetuned_models"
output_filename = "finetuned_models"

# 检查目录是否存在
if not os.path.exists(models_dir):
    print(f"❌ 错误：未找到目录 {models_dir}！请确保训练已成功完成。")
else:
    # 创建 zip 文件
    try:
        shutil.make_archive(output_filename, 'zip', models_dir)
        print(f"✅ 模型文件已打包为 {output_filename}.zip")
        print(f"   大小: {os.path.getsize(output_filename + '.zip') / 1024:.2f} KB")

        # 列出包含的文件
        print(f"\n📁 包含的文件:")
        !ls -lh {models_dir}

        print("\n⏳ 开始下载...")
        files.download(output_filename + '.zip')

        print("\n✅ 下载完成！")
        print("=" * 60)

    except Exception as e:
        print(f"\n❌ 打包或下载失败: {e}")

📦 打包微调模型文件...
✅ 模型文件已打包为 finetuned_models.zip
   大小: 4151210.21 KB

📁 包含的文件:
total 8.0K
drwxr-xr-x 12 root root 4.0K Nov  5 01:17 assignment2-deepseek-dow30_202511042216
drwxr-xr-x 12 root root 4.0K Nov  4 22:05 assignment2-llama31-dow30_202511041921

⏳ 开始下载...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 下载完成！
